## Partie 4 : Préparation des données pour le Dashboard Power BI / Looker Studio

> **Objectif** : Exporter les agrégats et tables nécessaires à la construction du dashboard de pilotage de campagne.

---
**Auteur** : Malcom Closse — Marketing Data Analyst  
**Prérequis** : Avoir exécuté les Parties 1, 2 & 3 (`data/processed/bank_scored.csv` requis)

In [ ]:
import pandas as pd
import numpy as np
import os

TABLES_PATH = os.path.join("..", "outputs", "tables")
os.makedirs(TABLES_PATH, exist_ok=True)

df = pd.read_csv(os.path.join("..", "data", "processed", "bank_scored.csv"))

MONTH_ORDER = [
    "jan",
    "feb",
    "mar",
    "apr",
    "may",
    "jun",
    "jul",
    "aug",
    "sep",
    "oct",
    "nov",
    "dec",
]
df["month"] = pd.Categorical(df["month"], categories=MONTH_ORDER, ordered=True)

print(f" Dataset chargé : {df.shape[0]:,} lignes")

✅ Dataset chargé : 11,162 lignes


### Table 1 — KPI globaux

In [9]:
kpi = pd.DataFrame(
    [
        {
            "total_contacts": len(df),
            "total_conversions": int(df["subscribed_bin"].sum()),
            "taux_conversion_pct": round(df["subscribed_bin"].mean() * 100, 2),
            "balance_moyen_yes": round(
                df[df["subscribed"] == "yes"]["balance"].mean(), 0
            ),
            "balance_moyen_no": round(
                df[df["subscribed"] == "no"]["balance"].mean(), 0
            ),
            "duree_moy_appel_s": round(df["duration"].mean(), 0),
            "contacts_moy_camp": round(df["campaign"].mean(), 2),
            "pct_priorite1": round(
                df["priorite"].str.contains("PRIORITÉ 1").mean() * 100, 1
            ),
        }
    ]
)
kpi.to_csv(os.path.join(TABLES_PATH, "db_kpi_globaux.csv"), index=False)
print(" db_kpi_globaux.csv")
display(kpi)

✅ db_kpi_globaux.csv


,total_contacts,total_conversions,taux_conversion_pct,balance_moyen_yes,balance_moyen_no,duree_moy_appel_s,contacts_moy_camp,pct_priorite1
0,11162,5289,47.38,1804.0,1280.0,372.0,2.51,45.5


### Table 2 — Conversion par mois

In [10]:
t_month = (
    df.groupby("month", observed=True)
    .agg(nb_contacts=("subscribed_bin", "count"), conversions=("subscribed_bin", "sum"))
    .assign(taux_pct=lambda x: (x["conversions"] / x["nb_contacts"] * 100).round(2))
    .reset_index()
)
t_month["month_num"] = t_month["month"].map(
    {m: i + 1 for i, m in enumerate(MONTH_ORDER)}
)
t_month.to_csv(os.path.join(TABLES_PATH, "db_conversion_mois.csv"), index=False)
print(" db_conversion_mois.csv")
display(t_month)

✅ db_conversion_mois.csv


,month,nb_contacts,conversions,taux_pct,month_num
0,jan,344,142,41.28,1
1,feb,776,441,56.83,2
2,mar,276,248,89.86,3
3,apr,923,577,62.51,4
4,may,2824,925,32.75,5
5,jun,1222,546,44.68,6
6,jul,1514,627,41.41,7
7,aug,1519,688,45.29,8
8,sep,319,269,84.33,9
9,oct,392,323,82.40,10


### Table 3 — Conversion par job

In [11]:
t_job = (
    df.dropna(subset=["job"])
    .groupby("job")
    .agg(
        nb_contacts=("subscribed_bin", "count"),
        conversions=("subscribed_bin", "sum"),
        balance_moy=("balance", "mean"),
    )
    .assign(taux_pct=lambda x: (x["conversions"] / x["nb_contacts"] * 100).round(2))
    .sort_values("taux_pct", ascending=False)
    .reset_index()
)
t_job.to_csv(os.path.join(TABLES_PATH, "db_conversion_job.csv"), index=False)
print(" db_conversion_job.csv")
display(t_job)

✅ db_conversion_job.csv


,job,nb_contacts,conversions,balance_moy,taux_pct
0,student,360,269,1500.783333,74.72
1,retired,778,516,2417.250643,66.32
2,unemployed,357,202,1314.719888,56.58
3,management,2566,1301,1793.663679,50.70
4,admin.,1334,631,1195.866567,47.30
5,self-employed,405,187,1865.372840,46.17
6,technician,1823,840,1556.294569,46.08
7,services,923,369,1081.171181,39.98
8,housemaid,274,109,1366.160584,39.78
9,entrepreneur,328,123,1621.942073,37.50


### Table 4 — Saturation des contacts

In [12]:
t_sat = (
    df.groupby("campaign")
    .agg(nb_contacts=("subscribed_bin", "count"), taux_pct=("subscribed_bin", "mean"))
    .assign(taux_pct=lambda x: (x["taux_pct"] * 100).round(2))
    .query("nb_contacts >= 50 and campaign <= 15")
    .reset_index()
)
t_sat.to_csv(os.path.join(TABLES_PATH, "db_saturation.csv"), index=False)
print(" db_saturation.csv")
display(t_sat)

✅ db_saturation.csv


,campaign,nb_contacts,taux_pct
0,1,4798,53.38
1,2,3028,46.27
2,3,1321,46.78
3,4,771,41.12
4,5,378,36.77
5,6,265,34.72
6,7,139,33.81
7,8,128,25.00
8,9,72,29.17
9,10,52,26.92


### Table 5 — Répartition des priorités

In [13]:
t_prio = (
    df.groupby("priorite")
    .agg(
        nb_clients=("subscribed_bin", "count"),
        taux_pct=("subscribed_bin", "mean"),
        score_moy=("propension_score", "mean"),
        balance_moy=("balance", "mean"),
    )
    .assign(
        taux_pct=lambda x: (x["taux_pct"] * 100).round(2),
        pct_total=lambda x: (x["nb_clients"] / x["nb_clients"].sum() * 100).round(1),
    )
    .reset_index()
)
t_prio.to_csv(os.path.join(TABLES_PATH, "db_priorites.csv"), index=False)
print(" db_priorites.csv")
display(t_prio)

✅ db_priorites.csv


,priorite,nb_clients,taux_pct,score_moy,balance_moy,pct_total
0,PRIORITÉ 1 — Contacter en urgence,5081,86.38,0.830234,1870.597520,45.5
1,PRIORITÉ 2 — À qualifier,1274,48.59,0.483240,1527.553375,11.4
2,PRIORITÉ 3 — Nurturing,1171,19.21,0.239613,1261.238258,10.5
3,PRIORITÉ 4 — Ne pas contacter,3636,1.54,0.047881,1136.971397,32.6


### Table 6 — Lift par décile

In [14]:
global_rate = df["subscribed_bin"].mean()
t_lift = (
    df.groupby("score_decile", observed=True)
    .agg(
        nb_clients=("subscribed_bin", "count"),
        taux_pct=("subscribed_bin", "mean"),
        score_moy=("propension_score", "mean"),
    )
    .assign(
        taux_pct=lambda x: (x["taux_pct"] * 100).round(2),
        lift=lambda x: (x["taux_pct"] / 100 / global_rate).round(3),
    )
    .reset_index()
)
t_lift.to_csv(os.path.join(TABLES_PATH, "db_lift_deciles.csv"), index=False)
print(" db_lift_deciles.csv")
display(t_lift)

✅ db_lift_deciles.csv


,score_decile,nb_clients,taux_pct,score_moy,lift
0,D1,1117,0.18,0.017279,0.004
1,D10,1117,98.03,0.944973,2.069
2,D2,1116,0.63,0.032322,0.013
3,D3,1116,2.24,0.072143,0.047
4,D4,1116,13.44,0.189066,0.284
5,D5,1116,38.17,0.397582,0.806
6,D6,1116,62.01,0.609523,1.309
7,D7,1116,78.32,0.751954,1.653
8,D8,1116,86.56,0.833959,1.827
9,D9,1116,94.27,0.889359,1.989


### Table 7 — Dataset complet scoré (pour Power BI)

In [16]:
cols_export = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "balance",
    "housing",
    "loan",
    "contact",
    "month",
    "duration",
    "campaign",
    "poutcome",
    "subscribed",
    "subscribed_bin",
    "balance_segment",
    "segment",
    "propension_score",
    "score_decile",
    "priorite",
]
cols_export = [c for c in cols_export if c in df.columns]

df[cols_export].to_csv(os.path.join(TABLES_PATH, "db_dataset_complet.csv"), index=False)
print("db_dataset_complet.csv")
print(f"   {len(df):,} lignes × {len(cols_export)} colonnes")
print("\n Toutes les tables Power BI exportées dans outputs/tables/")
print("\n Récap des fichiers :")
for f in sorted(os.listdir(TABLES_PATH)):
    if f.startswith("db_"):
        size = os.path.getsize(os.path.join(TABLES_PATH, f))
        print(f"   {f:<40} {size / 1024:.1f} KB")

db_dataset_complet.csv
   11,162 lignes × 20 colonnes

 Toutes les tables Power BI exportées dans outputs/tables/

 Récap des fichiers :
   db_conversion_job.csv                    0.5 KB
   db_conversion_mois.csv                   0.3 KB
   db_dataset_complet.csv                   1673.9 KB
   db_kpi_globaux.csv                       0.2 KB
   db_lift_deciles.csv                      0.4 KB
   db_priorites.csv                         0.4 KB
   db_saturation.csv                        0.1 KB
